# AI Assistant

In [3]:
pip install transformers torchaudio soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 4.3 MB/s  0:00:00

   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   ---------------------------------------- 0/2 [torchaudio]
   -------------------- ------------------- 1/2 [soundfile]
   ---------------------------------------- 2/2 [soundfile]

Note: you may need to restart the kernel to use updated packages.

In [2]:
pip install accelerate

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load the environment variables from the .env file in the src directory
load_dotenv(dotenv_path="../.env") # Adjust the path if your notebook is in the root directory

# Get your key from the environment (you named it HF_API_KEY in your .env)
hf_token = os.getenv("HF_API_KEY")

if hf_token:
    print("Logging into Hugging Face...")
    login(token=hf_token)
else:
    print("HF_API_KEY not found. Please check your .env file.")

c:\Users\juanr\OneDrive\Escritorio\SYNTECXHUB\personalVoiceAssistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Logging into Hugging Face...


In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient, hf_hub_download

load_dotenv(dotenv_path="../.env")

print("Downloading sample audio file...")
audio_file_path = hf_hub_download(
    repo_id="ibm-granite/granite-4.0-1b-speech", 
    filename="multilingual_sample.wav"
)

client = InferenceClient(token=os.getenv("HF_API_KEY"))

try:
    print("Sending audio to the cloud...")
    result = client.automatic_speech_recognition(
        audio="../audios/firstTry.wav", 
        # 👇 Change to a supported model for the free cloud API!
        model="openai/whisper-large-v3-turbo" 
    )
    print("\nCloud STT Output:", result.text)
except Exception as e:
    print("Exact Error:", repr(e))

Sending audio to the cloud...

Cloud STT Output:  Hola hola hola hola hola hola hola


In [9]:
from openai import OpenAI

client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=os.getenv("HF_API_KEY"))
response = client.chat.completions.create(model="meta-llama/Llama-3.1-8B-Instruct", messages= [
    {"role": "system", "content": "You are a friendly assistant who answers everything you get asked."},
    {"role": "user", "content": result.text}
], 
max_tokens=50)
rp = response.choices[0].message.content
print(rp)

Hola hola hola hola también! *sonrisa* ¿En qué puedo ayudarte hoy?


### Text - to - speech

In [12]:
from IPython.display import Audio, display
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

client = InferenceClient(token=os.getenv("HF_API_KEY"))

text_to_speak = "Hello there! I am your new personal voice assistant."

try:
    print("Generating audio from text...")
    
    # 1. Call the text-to-speech API with the correct variable
    audio_bytes = client.text_to_speech(
        text=rp,  # ← Fixed: was 'rp'
        # Try a more compatible TTS model
        model="microsoft/speecht5_tts"  # or "espnet/kan-bayashi_ljspeech_vits"
    )
    
    # 2. Save the audio to a file
    output_path = "../audios/assistant_response.wav"
    os.makedirs("../audios", exist_ok=True)  # Create directory if needed
    with open(output_path, "wb") as f:
        f.write(audio_bytes)
        
    print(f"\nSuccess! Audio saved to: {output_path}")
    
    # 3. Play the audio in the notebook
    display(Audio(audio_bytes, autoplay=True))
    
except Exception as e:
    print("Exact Error:", repr(e))
    print("Error type:", type(e).__name__)
    import traceback
    traceback.print_exc()  # Get full error trace

Generating audio from text...
Exact Error: StopIteration()
Error type: StopIteration


Traceback (most recent call last):
  File "C:\Users\juanr\AppData\Local\Temp\ipykernel_26740\2984711369.py", line 15, in <module>
    audio_bytes = client.text_to_speech(
        text=rp,  # ← Fixed: was 'rp'
        # Try a more compatible TTS model
        model="microsoft/speecht5_tts"  # or "espnet/kan-bayashi_ljspeech_vits"
    )
  File "c:\Users\juanr\OneDrive\Escritorio\SYNTECXHUB\personalVoiceAssistant\.venv\Lib\site-packages\huggingface_hub\inference\_client.py", line 2853, in text_to_speech
    provider_helper = get_provider_helper(self.provider, task="text-to-speech", model=model_id)
  File "c:\Users\juanr\OneDrive\Escritorio\SYNTECXHUB\personalVoiceAssistant\.venv\Lib\site-packages\huggingface_hub\inference\_providers\__init__.py", line 256, in get_provider_helper
    provider = next(iter(provider_mapping)).provider
               ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
StopIteration
